<a href="https://www.kaggle.com/code/abhishekgodara/translation-akkadian-english-score-34-2?scriptVersionId=292109457" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================
# Deep Past Initiative – High-Score Inference
# ByT5 Checkpoint Averaging (34+ baseline)
# ============================================

import re
import gc
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =========================
# Config
# =========================
TEST_DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"

MODEL1_PATH = "/kaggle/input/byt5-base-big-data2"
MODEL2_PATH = "/kaggle/input/byt5-akkadian-model"
MODEL3_PATH = "/kaggle/input/train-gap-all-2/byt5-base-akkadian_gap_setence2"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PREFIX = "translate Akkadian to English: "
MAX_SOURCE_LEN = 512
MAX_NEW_TOKENS = 512

BATCH_SIZE = 10
NUM_BEAMS = 10
LENGTH_PENALTY = 1.1
EARLY_STOPPING = True

USE_AMP = False  # fp32 decoding = more stable scores

# =========================
# Gap normalization (VERY IMPORTANT)
# =========================
def replace_gaps(text):
    if pd.isna(text):
        return text
    text = str(text)
    text = re.sub(r'\.3(?:\s+\.3)+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'\.3(?:\s+\.3)+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'xx', '<gap>', text)
    text = re.sub(r' x ', ' <gap> ', text)
    text = re.sub(r'……', '<big_gap>', text)
    text = re.sub(r'\.\.\.\.\.\.', '<big_gap>', text)
    text = re.sub(r'…', '<big_gap>', text)
    text = re.sub(r'\.\.\.', '<big_gap>', text)
    return text

# =========================
# Load test data
# =========================
test_df = pd.read_csv(TEST_DATA_PATH)
test_df["transliteration"] = test_df["transliteration"].apply(replace_gaps)

# =========================
# Load models
# =========================
print("Loading models...")

m1 = AutoModelForSeq2SeqLM.from_pretrained(MODEL1_PATH)
m2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)
m3 = AutoModelForSeq2SeqLM.from_pretrained(MODEL3_PATH)

sd1, sd2, sd3 = m1.state_dict(), m2.state_dict(), m3.state_dict()

# =========================
# Weighted checkpoint averaging
# =========================
perf1, perf2, perf3 = 0.99, 1.1, 0.40
total = perf1 + perf2 + perf3
w1, w2, w3 = perf1/total, perf2/total, perf3/total

print(f"Weights → w1={w1:.3f}, w2={w2:.3f}, w3={w3:.3f}")

final_sd = sd2.copy()
for k in final_sd:
    if k in sd1 and k in sd3:
        final_sd[k] = w1 * sd1[k] + w2 * sd2[k] + w3 * sd3[k]
    elif k in sd1:
        final_sd[k] = w1 * sd1[k] + (w2 + w3) * sd2[k]
    elif k in sd3:
        final_sd[k] = w3 * sd3[k] + (w1 + w2) * sd2[k]

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)
model.load_state_dict(final_sd)
model.to(DEVICE).eval().float()

tokenizer = AutoTokenizer.from_pretrained(MODEL2_PATH)

del m1, m2, m3, sd1, sd2, sd3
gc.collect()
torch.cuda.empty_cache()

# =========================
# Dataset + DataLoader
# =========================
class InferenceDataset(Dataset):
    def __init__(self, df):
        self.ids = df["id"].tolist()
        self.texts = [PREFIX + t for t in df["transliteration"].astype(str)]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.ids[idx], self.texts[idx]

def collate_fn(batch):
    ids, texts = zip(*batch)
    enc = tokenizer(
        list(texts),
        max_length=MAX_SOURCE_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    return list(ids), enc["input_ids"], enc["attention_mask"]

loader = DataLoader(
    InferenceDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

# =========================
# Inference
# =========================
all_ids, all_pred = [], []

with torch.inference_mode():
    for ids, input_ids, attention_mask in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,
            length_penalty=LENGTH_PENALTY,
            early_stopping=EARLY_STOPPING,
        )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        decoded = [d.strip() if d.strip() else "broken text" for d in decoded]

        all_ids.extend(ids)
        all_pred.extend(decoded)

# =========================
# Save submission
# =========================
submission = pd.DataFrame({
    "id": all_ids,
    "translation": all_pred
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv saved")
print(submission.head())

Loading models...


2026-01-15 21:58:25.264774: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768514305.461121      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768514305.517310      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768514305.991630      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768514305.991669      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768514305.991672      55 computation_placer.cc:177] computation placer alr

Weights → w1=0.412, w2=0.420, w3=0.168
✅ submission.csv saved
   id                                        translation
0   0  Thus Kanesh colony, say to the <big_gap> of ou...
1   1  In the tablet of the City you wrote to me in t...
2   2  Just as you hear our letter, he has given eith...
3   3  I sent our certified tablets to every single d...
